In [ ]:
# from numba import cuda
import paicos as pa
import numpy as np
import cupy as cp
import turbocluster as tc
import math
from numba import cuda
import nvtx
import finufft
from paicos.util import get_index_of_radial_range as RadialRange
import os

pa.settings.strict_units = False

snap = pa.Snapshot('/llust21/cosmo-plasm/zoom-simulations-arepo2/halo_0003/tng/zoom12/output', 
                   305, basename='snapshot')
center = snap.Cat.Group['GroupPos'][0]
R200 = snap.Cat.Group['Group_R_Crit200'][0]
widths = 2.0 * np.ones(3) * R200


filter_weight = None
filter_type = 'gaussian'
conversion_factor = 5.33645 ## between physical length scale and filter size (Eq.A6 of Perrone+2026)
physical_scale = 100 ## this is in kpc physical units!

kpc = pa.units.PaicosQuantity(1 * snap.h /snap.a, 'kpc small_a / small_h ', 
                              h=snap.h, a=snap.a, comoving_sim=True)

filter_length = physical_scale*kpc.to('arepo_length')/conversion_factor

idxR200 = RadialRange(snap["0_Coordinates"],
                          center, 0*R200, R200)


In [ ]:
import cmasher as cmr
%matplotlib widget
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.colors import LogNorm, Normalize, SymLogNorm
from matplotlib.patches import Circle, Rectangle
import matplotlib.patches as patches
from matplotlib.ticker import LogLocator, LogFormatter
from matplotlib.gridspec import GridSpec

grays = plt.cm.gray(np.linspace(0,1,10))
reds = plt.cm.Reds_r(np.linspace(0,1,10))
blues = plt.cm.Blues_r(np.linspace(0,1,10))
greens = plt.cm.Greens_r(np.linspace(0,1,10))
oranges = plt.cm.Oranges_r(np.linspace(0,1,10))

colors20b = plt.get_cmap('tab20b').colors
colors20c = plt.get_cmap('tab20c').colors

rdpu = plt.cm.RdPu(np.linspace(0,1,4))
pubu = plt.cm.PuBu(np.linspace(0,1,4))

plt.rcParams['font.size'] = 12
plt.rcParams['axes.labelsize'] = 12
plt.rcParams['lines.linewidth'] = 1.7
plt.rcParams['axes.linewidth'] = 1.5
plt.rcParams['xtick.labelsize']= 12
plt.rcParams['ytick.labelsize']= 12
plt.rcParams['xtick.minor.size']= 4.0
plt.rcParams['ytick.minor.size']= 4.0
plt.rcParams['xtick.major.size']= 5.0
plt.rcParams['ytick.major.size']= 5.0
plt.rcParams['xtick.minor.width']= 0.7
plt.rcParams['ytick.minor.width']= 0.7
plt.rcParams['xtick.major.width']= 1.
plt.rcParams['ytick.major.width']= 1.

plt.rcParams['xtick.direction']= 'in'
plt.rcParams['ytick.direction']= 'in'

In [ ]:
sf = tc.SmoothingFilter(snap, center, widths, npix=256, orientation=None, 
                                    search_radius=filter_length, gauss_multiplier=6)

turb = {}

In [ ]:
# shear_matrix_Favre = tc.compute_shear_matrix(snap, sf, snap['0_Velocities'], '0_Density', 
#                                        filter_length,  filter_type,  
#                                recycle=False, snap_out=turb, selection=idxR200)

In [ ]:
# shear_matrix = tc.compute_shear_matrix(snap, sf, snap['0_Velocities'], None, 
#                                        filter_length,  filter_type,  
#                                recycle=False, snap_out=turb, selection=idxR200)

In [ ]:
# mu2_matrix = tc.compute_mu2_tensor(snap, sf, snap['0_Velocities'], None, 
#                                    filter_length, 
#                               filter_type, recycle=False, snap_out=turb, 
#                                 selection=idxR200)

In [ ]:
# mu2_matrix_Favre = tc.compute_mu2_tensor(snap, sf, snap['0_Velocities'], '0_Density', 
#                                    filter_length, 
#                               filter_type, recycle=False, snap_out=turb, 
#                                 selection=idxR200)

In [ ]:
SGS_stress = tc.compute_SGS_stress(snap, sf, snap['0_Velocities'], '0_Density', 
                                   '0_Density', filter_length, filter_type, recycle=False, 
                                   snap_out=turb, selection=idxR200)

In [ ]:
widths_slicer = 2.0 * np.ones(3) * R200
widths_slicer[2] = 0.
slicer = pa.Slicer(snap, center, widths_slicer, 'z', npix=1024)

extent = slicer.centered_extent.to('Mpc').to_physical

In [ ]:
# mu2_slice = slicer.slice_variable(mu2_matrix[:,0,0]).to_physical.to('km2/s2')
# mu2favre_slice = slicer.slice_variable(mu2_matrix_Favre[:,0,0]).to_physical.to('km2/s2')

In [ ]:

# vmax = 2e-1*np.nanmax(mu2_slice.value)
# # vmin = 1e-3*vmax
# norm = Normalize(vmin=-vmax,vmax=vmax)

# # vmax = 1e-1*np.nanmax(grad_logpx)
# # vmin = -vmax
# # norm2 = Normalize(vmin=vmin,vmax=vmax)
# # norm2 = SymLogNorm(linthresh=3e-3*vmax, linscale=3.0,
# #                         vmin=-8e-1*vmax, vmax=8e-1*vmax, base=10)

# cmap = cmr.prinsenvlag_r
# # cmap2 = cmr.rainforest
# # cmap2 = cmr.prinsenvlag_r
# # cmap2 = cmr.eclipse
# # cmap2 = cmr.ghostlight

# fig, axes = plt.subplots(1,2, figsize=(10,6),sharex=True, sharey=True)

# im00 = axes[0].imshow(
#     mu2_slice.value, origin='lower', cmap=cmap, norm=norm, extent=extent.value)

# im01 = axes[1].imshow(
#     mu2favre_slice.value, origin='lower', cmap=cmap, norm=norm, extent=extent.value)


# axes[0].set_title(r'$\mu_2$')
# axes[1].set_title(r'$\mu_2$ Favre')


# for i in range(2):
#     axes[i].set_xlabel(extent.label('x'))

# axes[0].set_ylabel(extent.label('y'))
# # axes[1,0].set_ylabel(extent.label('y'))


# cbar00 = fig.colorbar(im00, orientation='horizontal',shrink=0.9)
# cbar00.set_label(mu2_slice.label('\\mu_2 (u, u)'))

# cbar01 = fig.colorbar(im01, orientation='horizontal',shrink=0.9)
# cbar01.set_label(mu2_slice.label('\\mu_2 (u, u)'))


# fig.subplots_adjust(top=0.9,
# bottom=0.05,
# left=0.07,
# right=0.95,
# hspace=0.15,
# wspace=0.00)

# fig.suptitle('comparison favre (fixed filter)',y=0.98)
# plt.savefig('./comparison_mu2_favre_l_100.pdf', dpi=400)

# # plt.show()

In [ ]:
SGS_stress.uq.to_physical.to('erg / (Myr km3)')

In [ ]:
SGS_stress

In [ ]:
SGS_slice = slicer.slice_variable(SGS_stress).to_physical.to('erg / (Myr km3)')

In [ ]:

vmax = 8e-1*np.max(np.abs(SGS_slice.value))
# vmin = 1e-3*vmax
# norm = Normalize(vmin=-vmax,vmax=vmax)

# vmax = 1e-1*np.nanmax(grad_logpx)
# vmin = -vmax
# norm2 = Normalize(vmin=vmin,vmax=vmax)
norm2 = SymLogNorm(linthresh=3e-4*vmax, linscale=2.0,
                        vmin=-8e-1*vmax, vmax=8e-1*vmax, base=10)

cmap = cmr.prinsenvlag_r
# cmap2 = cmr.rainforest
# cmap2 = cmr.prinsenvlag_r
# cmap2 = cmr.eclipse
# cmap2 = cmr.ghostlight

fig, axes = plt.subplots(figsize=(8,6))

im00 = axes.imshow(
    SGS_slice.value, origin='lower', cmap=cmap, norm=norm2, extent=extent.value)

axes.set_title(r'SGS')


axes.set_xlabel(extent.label('x'))
axes.set_ylabel(extent.label('y'))



cbar00 = fig.colorbar(im00, orientation='horizontal',shrink=0.9)
cbar00.set_label(SGS_slice.label('sgs'))



fig.subplots_adjust(top=0.9,
bottom=0.05,
left=0.07,
right=0.95,
hspace=0.15,
wspace=0.00)

fig.suptitle('Subgridscale stress (fixed filter)',y=0.98)
plt.savefig('./comparison_sgs_favre_l_100.pdf', dpi=400)

# plt.show()